# 3D Fluorescence Nuclei Segmentation (v1.6.41)

Segments and quantifies nuclei and cell markers in 3D (or 2D) fluorescence images.

> Python 3.10–3.14. Install: `uv pip install -r requirements.txt --override overrides.txt`. On 3.14, StarDist and `.czi` files are unavailable.

**How to use**
1. Set file, ROI, experiment name and sizes in **Cell 3**, exports in **Cell 4**.
2. Run Cell 5, fill in the stain table and click **Confirm**.
3. Run the rest. The processing steps of Cells 9-14 are optional (see Section 4).

**Segmentation settings.** The segmentation uses settings validated against manual ground truths (`settings/optimal_settings_v1.6.41.json`; nuclei mean F1 0.56, F1@0.5 0.77 on six annotated images, each scored as a new image). Some of them are adapted to each image (Cell 8):
- the **NUCLEI contrast**, from the channel's own histogram;
- the **nucleus size**: `nuclei_diameter` in Cell 3 is only a starting value, the typical nucleus is measured in the image and the sizes adapted to it;
- the **settings that depend on the image quality**, read off the channel histograms (background, noise, signal-to-noise ratio): the nuclei edge fraction is lower in a noisy NUCLEI channel, higher in a clean one.

The cell bodies (Cell 18) have their own settings: the cytoplasm / membrane stains are thresholded on their own and every cell is completed by the dim rim of its body (`cyto_grow_um`).

Every other setting is fixed. To change one anyway, name it in `custom_settings` (Cell 4) and re-run from Cell 4; a setting named there is not adapted to the image. All settings, with their validated values, are listed in `settings/optimal_settings_v1.6.41.json`; the validation is described in `README.md`.

**Methods:** watershed (default, the validated one), Cellpose (`trig_cellpose`), StarDist (`trig_stardist`). Cell shape: nuclei-seeded watershed or Cellpose (`trig_cellpose_cyto`).


# 1. Import Libraries and Helpers

**Cell 1** — Import libraries (prints the Python version).

In [ ]:
from helpers.notebook_setup_helpers import load_nuclei_notebook_setup
globals().update(load_nuclei_notebook_setup())

**Cell 2** — Load helper functions. Re-run after editing `helpers/notebook_helpers.py`.

In [ ]:
from helpers.notebook_helpers import reload_helpers
reload_helpers()

# 2. Inputs and Setup

The only cells to edit for a new dataset.

**Cell 3** — Settings. A folder as `input_file` runs **batch mode** (every image gets its own NUCLEI contrast, nucleus size and histogram-adapted settings). Each setting is explained in line.

In [ ]:
# --- Input ---
input_file = 'path/to/your_image_file.nd2'  # file (.nd2/.tif/.czi/.lif/.oib) or folder (batch)
ROI = [0, 0, 0, 0, 0, 0]                    # [x0, x1, y0, y1, z0, z1]; 0 = full range
interactive_roi = False                     # True = draw ROI in napari

# --- Experiment ---
name_setup = 'your_experiment_name'  # prefix of saved files
use_setup = True                     # reuse saved stain table and contrast
calibrated_contrast = True           # marker channels: True = contrast from each channel's histogram (calibrated, no napari window), False = set it yourself in napari (Cell 8)
                                     # (the NUCLEI contrast is always set from its histogram, Cell 8)

# --- Sizes ---
nuclei_diameter = 8   # um, approximate: the typical nucleus is measured in the image (Cell 8)
cell_diameter = 20    # um

# --- Segmentation method (all False = watershed, the validated method) ---
trig_cellpose = False       # Cellpose for nuclei (GPU recommended)
trig_stardist = False       # StarDist for nuclei (not on Python 3.14)
trig_cellpose_cyto = False  # Cellpose for cell shape (needs CYTOPLASM channel)

# --- Quantification ---
multilabel = True                 # multi-marker combinations in summary
aggregate_distance_factor = 0.05  # cells closer than this x cell_diameter = one aggregate
pcm_thickness_factor = 0.05       # PCM shell thickness = this x cell_diameter

**Cell 4** — Exports and segmentation settings, used in single-file and batch mode. The segmentation settings are the validated ones; `custom_settings` changes one by name (e.g. `{"nuclei_edge_fraction": 0.3}`; names in `settings/optimal_settings_v1.6.41.json`).

In [ ]:
# --- 3D exports (skipped for 2D images) ---
export_vtk = True      # VTK volumes for ParaView (slow)
export_stl = True      # STL mesh per marker (slow)
export_fea = True      # FEA mesh, Abaqus .inp (slow)

# --- Segmentation settings: validated, normally left as they are ---
custom_settings = {}   # e.g. {"nuclei_edge_fraction": 0.3}; {} = all validated values

from helpers.optimal_settings import (
    load_optimal_settings, adapt_to_histograms, set_nuclei_contrast, adapt_to_image, correct_depth, remove_haze,
    processing_settings, segmentation_settings, cytoplasm_settings, batch_settings,
    settings_for_report)
opt = load_optimal_settings(custom_settings, version="1.6.41")

**Cell 5** — Load image and stain table. Single-plane images are handled as 2D (`is_2d`).

Fill in the table: channel, color and marker per condition; tick **Intracellular?** or **Membrane?** for cytoplasm stains. Click **Confirm**.

In [ ]:
batch_mode = is_batch_input(input_file)
file_meta = None
is_2d = False  # set from the loaded image
stain_table_path = f"{name_setup}_stain_dict.json" if use_setup else None

if batch_mode:
    print(f"Batch mode: {len(list_image_files(input_file))} image(s) will be "
          f"processed by Cell 6.")
else:
    if interactive_roi:
        ROI = select_roi_interactively(input_file, ROI, napari_module=napari,
                                       stain_table_path=stain_table_path)
    (meta, img, r_X, r_Y, r_Z, file_meta, ROI_print,
     cyto_factor, PCM_factor, zooms, used_lazy_loading) = initialize_dataset(
        input_file, ROI,
        nuclei_diameter=nuclei_diameter,
        cell_diameter=cell_diameter,
        scale_factor=opt["scale_factor"],
        zoom_factors=opt["zoom_factors"],
    )
    is_2d = img.shape[0] == 1
    print("2D image: segmentation in 2D, sizes as areas [um2], 3D exports skipped."
          if is_2d else f"3D image: {img.shape[0]} Z planes.")

channel_list = print_channel_list(input_file, file_meta=file_meta)
stain_editor = edit_stain_dict(channel_list, name_setup=name_setup, use_saved=use_setup)

**Cell 6** — Read the stain table. In batch mode, processes the whole folder (to `<folder>_output/`) with the settings of Cells 3–4 and stops.

In [ ]:
stain_dict, cyto_markers = stain_editor.result()
membrane_markers = stain_editor.membrane_markers  # also in cyto_markers
print(f"stain_dict = {stain_dict}")
print(f"cyto_markers = {cyto_markers}")
print(f"membrane_markers = {membrane_markers}")

if batch_mode:
    batch_summary = run_batch_folder(
        input_file,
        roi_coords=ROI,
        stain_dict=stain_dict,
        cyto_markers=cyto_markers,
        membrane_markers=membrane_markers,
        name_setup=name_setup,
        use_setup=use_setup,
        calibrated_contrast=calibrated_contrast,
        interactive_roi=interactive_roi,
        nuclei_diameter=nuclei_diameter,
        cell_diameter=cell_diameter,
        trig_cellpose=trig_cellpose,
        trig_stardist=trig_stardist,
        trig_cellpose_cyto=trig_cellpose_cyto,
        multilabel=multilabel,
        aggregate_distance_factor=aggregate_distance_factor,
        pcm_thickness_factor=pcm_thickness_factor,
        export_vtk=export_vtk,
        export_stl=export_stl,
        export_fea=export_fea,
        **batch_settings(opt),  # validated segmentation settings (Cell 4)
        settings=settings,
        napari_module=napari,
        progress=tqdm,
    )
    display(batch_summary)
    raise BatchCompleted("Batch processing finished -- the remaining cells are for single-file mode only.")
else:
    print("Single-file mode -- continue with the next cells.")

# 3. Preparation and Preview

**Cell 7** — Prepare the image stack and preview the channels in napari.

In [ ]:
(im_final_stack, nuclei_radius, cell_radius, nuclei_volume, cell_volume,
 stain_df, viewer_0) = prepare_and_preview(
    img,
    nuclei_diameter, cell_diameter,
    stain_dict, file_meta,
    napari_module=napari, r_xyz=(r_X, r_Y, r_Z), progress=tqdm,
)

**Cell 8** — Contrast and gamma per channel: reused from `<name_setup>_setup.csv` if it exists, otherwise calibrated from each channel's histogram (`calibrated_contrast = True`, the default, no napari window) or set yourself in napari (`calibrated_contrast = False`). The settings that depend on the image quality are then read off the channel histograms (printed with their values), the NUCLEI contrast is set from the channel's own histogram (the rule the segmentation was validated with), and the typical nucleus is measured in the image and `nuclei_diameter`, the resolution and the µm sizes are adapted to it. The stain settings table is shown once the napari window is closed.

In [ ]:
stain_df, stain_complete_df, original_stain_complete_df = prepare_stain_settings(
    im_final_stack['Original image'],
    stain_df=stain_df,
    name_setup=name_setup,
    use_setup=use_setup,
    calibrated_contrast=calibrated_contrast,
    suggest_contrast=opt["suggest_contrast"],
    settings=settings,
    napari_module=napari,
    r_xyz=(r_X, r_Y, r_Z),
    progress=tqdm,
)
# settings that follow the channel histograms (nuclei edge fraction)
adapt_to_histograms(im_final_stack['Original image'], stain_complete_df, opt)
set_nuclei_contrast(im_final_stack['Original image'], stain_complete_df,
                    original_stain_complete_df, opt)
display(stain_complete_df)

nuclei_diameter, zooms, cyto_factor, adapt_changes = adapt_to_image(
    im_final_stack['Original image'], stain_complete_df, (r_Z, r_Y, r_X),
    nuclei_diameter, cell_diameter, opt, is_2d=is_2d)
im_final_stack = reset_processing_steps(im_final_stack)  # Cells 9-14 start from the original image

# 4. Image Processing and Segmentation

Run in order. The settings are the validated ones of Cell 4.

**Optional processing steps.** Cells 9–14 can each be skipped: a skipped cell passes its input on unchanged, and the next cell run works on the last image computed. E.g. skip Cell 11 to go from the resampled image (Cell 10) to the contrast (Cell 12), or skip Cells 9–14 to threshold the original image of Cell 8 directly. Cell 15 lists the skipped steps (they are also saved in the report). Without resampling (Cell 10) the image keeps its original voxel size. Skipping any step departs from the validated settings. To change which steps run, re-run from Cell 8.

**Cell 9** *(optional)* — Normalize each channel to 0–255 (after the depth correction, if switched on in `custom_settings`; off in the validated settings).

In [ ]:
im_final_stack = correct_depth(im_final_stack, stain_complete_df, opt, voxel_z=r_Z)
im_final_stack = run_normalize(
    im_final_stack,
    stain_complete_df=stain_complete_df,
)

**Cell 10** *(optional)* — Resample to isotropic voxels. Skipped: the image keeps its original voxel size.

In [ ]:
im_final_stack, r_zX, r_zY, r_zZ = run_resample(
    im_final_stack,
    stain_complete_df=stain_complete_df,
    zoom_factors=zooms,
    meta=meta,
    edge_mode=opt["resample_edge_mode"],  # keeps the last Z slice
)

**Cell 11** *(optional)* — Denoise (median filter), then remove the out-of-focus haze of the NUCLEI channel slice by slice.

In [ ]:
im_final_stack = run_denoise(
    im_final_stack,
    stain_complete_df=stain_complete_df,
)
im_final_stack = remove_haze(im_final_stack, stain_complete_df, opt, (r_zZ, r_zY, r_zX))

**Cell 12** *(optional)* — Apply contrast and gamma from Cell 8.

In [ ]:
im_final_stack = run_contrast_gamma(
    im_final_stack,
    stain_complete_df=stain_complete_df,
)

**Cell 13** *(optional)* — Gaussian smoothing.

In [ ]:
im_final_stack = run_smooth(
    im_final_stack,
    stain_complete_df=stain_complete_df,
    sigma=opt["sigma"],
)

**Cell 14** *(optional)* — Contrast enhancement (histogram equalization).

In [ ]:
im_final_stack = run_equalize(
    im_final_stack,
    stain_complete_df=stain_complete_df,
    num_plateaus=opt["num_plateaus"],
    plateau_factor=opt["plateau_factor"],
)

**Cell 15** — Threshold every channel (global + local), on the last image computed by Cells 9–14 (the skipped steps are listed first). The masks define nuclei, cytoplasm and marker positivity.

In [ ]:
im_final_stack, zooms, r_zX, r_zY, r_zZ, skipped_steps = complete_skipped_steps(
    im_final_stack, zoom_factors=zooms, r_xyz=(r_X, r_Y, r_Z))  # skipped Cells 9-14
im_final_stack = run_threshold(
    im_final_stack,
    stain_complete_df=stain_complete_df,
    nuclei_diameter=nuclei_diameter,
    cell_diameter=cell_diameter,
    r_zX=r_zX, r_zY=r_zY, r_zZ=r_zZ,
    threshold_method=opt["threshold_method"],
    global_threshold_weight=opt["global_threshold_weight"],
    sauvola_weight=opt["sauvola_weight"],
    progress=tqdm,
)

**Cell 16** — Export histograms and parameters to `<file>_histograms.xlsx`.

In [ ]:
output_path = export_channel_histograms(
    im_final_stack, stain_complete_df, input_file,
    ROI_print=ROI_print, lazy_loading_used=used_lazy_loading,
    name_setup=name_setup,
    nuclei_diameter=nuclei_diameter, cell_diameter=cell_diameter,
    cyto_factor=cyto_factor, pcm_thickness_factor=pcm_thickness_factor,
    zoom_factors=zooms, scale_factor=opt["scale_factor"],
    trig_cellpose=trig_cellpose, trig_stardist=trig_stardist,
    trig_cellpose_cyto=trig_cellpose_cyto,
    multilabel=multilabel, cyto_markers=cyto_markers,
    membrane_markers=membrane_markers,
    sigma=opt["sigma"], num_plateaus=opt["num_plateaus"],
    plateau_factor=opt["plateau_factor"], threshold_method=opt["threshold_method"],
    nuclei_split_config=opt["nuclei_split_config"],
    aggregate_distance_factor=aggregate_distance_factor,
    extra_params={'Skipped processing steps': '; '.join(skipped_steps) or 'none',
                  **settings_for_report(opt, adapt_changes)},
    progress=tqdm,
)

**Cell 17** — Segment nuclei with the chosen method and the validated settings: watershed, size refinement (merge fragments, re-split oversized nuclei), split of nuclei stacked in Z, nuclei the threshold missed added back, each nucleus edge redrawn relative to its own brightness. The recap table gives each nucleus's final label.

In [ ]:
im_segmentation_stack = segment_nuclei(
    im_final_stack,
    stain_df=stain_df,
    stain_complete_df=stain_complete_df,
    r_zxyz=(r_zX, r_zY, r_zZ),
    nuclei_diameter=nuclei_diameter,
    trig_stardist=trig_stardist,
    trig_cellpose=trig_cellpose,
    **segmentation_settings(opt),  # validated settings (Cell 4)
    progress=tqdm,
)

**Cell 18** — Segment cytoplasm: cytoplasm/membrane stains (thresholded with their own settings) assigned to nuclei, one cell per nucleus, completed by the dim rim of the cell body (`cyto_grow_um`). Nuclei without cytoplasm are excluded from marker analysis.

In [ ]:
im_segmentation_stack, stain_complete_df = segment_cytoplasm(
    im_final_stack,
    im_segmentation_stack=im_segmentation_stack,
    stain_df=stain_df,
    stain_complete_df=stain_complete_df,
    cyto_markers=cyto_markers,
    membrane_markers=membrane_markers,
    cyto_factor=cyto_factor,
    nuclei_diameter=nuclei_diameter,
    cell_diameter=cell_diameter,
    trig_cellpose_cyto=trig_cellpose_cyto,
    r_zxyz=(r_zX, r_zY, r_zZ),
    **cytoplasm_settings(opt),  # cell-body settings (and split_gamma)
    progress=tqdm,
)

**Cell 19** — Segment the pericellular matrix (PCM): a shell `pcm_thickness_factor` × `cell_diameter` thick around each cell.

In [ ]:
im_segmentation_stack = segment_pcm(
    im_segmentation_stack=im_segmentation_stack,
    stain_df=stain_df,
    cell_diameter=cell_diameter,
    r_zxyz=(r_zX, r_zY, r_zZ),
    pcm_thickness_factor=pcm_thickness_factor,
)

**Cell 20** — Assign cells to the markers they are positive for.

In [ ]:
im_segmentation_stack = assign_channel_labels(
    im_final_stack,
    im_segmentation_stack=im_segmentation_stack,
    stain_df=stain_df,
    progress=tqdm,
)

**Cell 21** — Detect cell aggregates: cells closer than `aggregate_distance_factor` × `cell_diameter` form one aggregate.

In [ ]:
im_segmentation_stack = detect_aggregates(
    im_segmentation_stack,
    stain_df=stain_df,
    cell_diameter=cell_diameter,
    r_zxyz=(r_zX, r_zY, r_zZ),
    aggregate_distance_factor=aggregate_distance_factor,
)

**Cell 22** — View processing steps and segmentation in napari. The processing viewer shows the original image and only the steps of Cells 9–14 that were run.

In [ ]:
viewer_0, viewer_1 = view_processing_results(
    im_final_stack,
    im_segmentation_stack=im_segmentation_stack,
    stain_df=stain_df,
    stain_complete_df=stain_complete_df,
    r_xyz=(r_X, r_Y, r_Z),
    r_zxyz=(r_zX, r_zY, r_zZ),
    napari_module=napari,
    progress=tqdm,
)

# 5. Quantification and Analysis

Intensities are reported for **positive cells** and **all cells**.

**Cell 23** — Measure intensity, size and position of every cell (`labels_df`).

In [ ]:
percell_mean_df, percell_std_df = compute_percell_marker_intensity_df(
    im_segmentation_stack, im_final_stack,
    stain_complete_df=stain_complete_df,
    progress=tqdm,
)

labels_df = build_labels_df(
    im_segmentation_stack, im_final_stack,
    stain_complete_df=stain_complete_df,
    stain_df=stain_df,
    r_zX=r_zX, r_zY=r_zY, r_zZ=r_zZ,
    zooms=zooms,
    multilabel=multilabel,
    percell_mean_df=percell_mean_df,
    percell_std_df=percell_std_df,
    progress=tqdm,
    is_2d=is_2d,
)

**Cell 24** — Population summary per condition.

In [ ]:
print_population_summary(
    labels_df,
    stain_complete_df=stain_complete_df,
    stain_df=stain_df,
    progress=tqdm,
)

**Cell 25** — Full-resolution table for export.

In [ ]:
labels_full_df = build_full_labels_df(
    im_segmentation_stack, im_final_stack,
    stain_complete_df=stain_complete_df,
    r_zX=r_zX, r_zY=r_zY, r_zZ=r_zZ,
    zooms=zooms,
    percell_mean_df=percell_mean_df,
    percell_std_df=percell_std_df,
    progress=tqdm,
    is_2d=is_2d,
)

**Cell 26** — Per-nucleus intensity report (PDF).

In [ ]:
hist_data, intensity_ranges, fig, axes, x_grid = build_histogram_report(
    im_segmentation_stack, im_final_stack,
    stain_df=stain_df,
    stain_complete_df=stain_complete_df,
    input_file=input_file,
    show_plot=False,  # KDE plots go to the PDF only
    progress=tqdm,
)

**Cell 27** — Spatial distribution: cell counts (lines) and marker intensity (dots) along X, Y, Z.

In [ ]:
plot_spatial_distributions(
    labels_df,
    stain_complete_df=stain_complete_df,
    stain_df=stain_df,
    im_final_stack=im_final_stack,
    r_X=r_X, r_Y=r_Y, r_Z=r_Z,
    zoom_factors=zooms,
    percell_mean_df=percell_mean_df,
    progress=tqdm,
)

**Cell 28** — Size distribution of nuclei and cells.

In [ ]:
plot_size_distributions(
    labels_df,
    stain_complete_df=stain_complete_df,
    stain_df=stain_df,
    progress=tqdm,
)

**Cell 29** — Marker intensity vs cytoplasm size (flow-cytometry style).

In [ ]:
plot_marker_intensity_clouds(
    percell_mean_df,
    stain_complete_df=stain_complete_df,
    im_segmentation_stack=im_segmentation_stack,
    r_xyz=(r_zX, r_zY, r_zZ),
    stain_df=stain_df,
    progress=tqdm,
    input_file=input_file,
    is_2d=is_2d,
)

# 6. Export

3D exports are skipped for 2D images. Which ones run is set in Cell 4; the single-cell export is set in Cell 32.

**Cell 30** — VTK volumes for ParaView (`export_vtk`).

In [ ]:
if not export_vtk:
    print("VTK volumes skipped (export_vtk = False).")
elif is_2d:
    print("2D image: VTK volumes skipped (3D only).")
else:
    build_vtk_volumes(
        im_segmentation_stack,
        labels_full_df=labels_full_df,
        stain_complete_df=stain_complete_df,
        input_file=input_file,
        r_xyz=(r_X, r_Y, r_Z),
        zoom_factors=zooms,
        progress=tqdm,
    )

**Cell 31** — STL mesh per marker (`export_stl`).

In [ ]:
if not export_stl:
    print("STL meshes skipped (export_stl = False).")
elif is_2d:
    print("2D image: STL meshes skipped (3D only).")
else:
    export_marker_stl(
        im_segmentation_stack,
        stain_df=stain_df,
        stain_complete_df=stain_complete_df,
        input_file=input_file,
        r_xyz=(r_X, r_Y, r_Z),
        zoom_factors=zooms,
        progress=tqdm,
    )

**Cell 32** — Single-cell VTK export. Set `export_nuc_3D_vtk = True` in this cell and run it; it asks for the nucleus label (see the recap table of Cell 17).

In [ ]:
export_nuc_3D_vtk = False  # True = export one cell as VTK, asks for its label

if not export_nuc_3D_vtk:
    print("Single-cell VTK export skipped (export_nuc_3D_vtk = False).")
elif is_2d:
    print("2D image: single-cell VTK export skipped (3D only).")
else:
    nuc_label = int(input("Enter nucleus label number to export: "))
    export_nucleus_vtk_crop(
        nuc_label=nuc_label,
        im_segmentation_stack=im_segmentation_stack,
        im_final_stack=im_final_stack,
        stain_df=stain_df,
        input_file=input_file,
        r_zxyz=(r_zX, r_zY, r_zZ),
        zoom_factors=zooms,
    )

**Cell 33** — Excel report (`<file>_segmentation.xlsx`).

In [ ]:
output_path = export_quantification_to_excel(
    input_file,
    original_stain_complete_df,
    labels_full_df,
    progress=tqdm,
)

**Cell 34** — FEA mesh (Abaqus `.inp`; `export_fea`).

In [ ]:
if not export_fea:
    print("FEA mesh skipped (export_fea = False).")
elif is_2d:
    print("2D image: FEA mesh skipped (3D only).")
else:
    export_fea_mesh(
        im_segmentation_stack,
        input_file=input_file,
        r_zxyz=(r_zX, r_zY, r_zZ),
        progress=tqdm,
    )